In [1]:

# ============================================================
# STUDENT PRODUCTIVITY PREDICTION SYSTEM
# Pemeriksaan dataset dan perbandingan model klasifikasi
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# 1. MEMBACA DATASET
FILE_PATH = "/content/drive/MyDrive/sistemcerdas/data_mahasiswa.csv"
TARGET = "Produktivitas"
ID_COL = "ID_Mahasiswa"

df = pd.read_csv(FILE_PATH)

# Membersihkan nama kolom dari spasi di awal/akhir
df.columns = df.columns.str.strip()

print("=" * 60)
print("1. INFORMASI DATASET")
print("=" * 60)
print("Ukuran dataset:", df.shape)
print("\nLima baris pertama:")
display(df.head())

print("\nJumlah nilai kosong:")
print(df.isnull().sum())

if TARGET not in df.columns:
    raise ValueError(
        f"Kolom target '{TARGET}' tidak ditemukan. "
        f"Kolom tersedia: {df.columns.tolist()}"
    )

# Hapus baris yang targetnya kosong karena tidak bisa dilatih
df = df.dropna(subset=[TARGET]).copy()

print("\nDistribusi target:")
print(df[TARGET].value_counts())
print("\nPersentase target:")
print((df[TARGET].value_counts(normalize=True) * 100).round(2))

# 2. PEMERIKSAAN DUPLIKASI
print("\n" + "=" * 60)
print("2. PEMERIKSAAN DUPLIKASI")
print("=" * 60)

feature_cols = [
    c for c in df.columns
    if c not in [TARGET, ID_COL]
]

print("Jumlah baris:", len(df))
print(
    "ID unik:",
    df[ID_COL].nunique() if ID_COL in df.columns else
    "Kolom ID tidak ditemukan"
)

# Duplikasi kombinasi fitur, tanpa ID dan target
print(
    "Kombinasi fitur unik:",
    df[feature_cols].drop_duplicates().shape[0]
)

# Apakah kombinasi fitur yang sama memiliki label berbeda?
label_counts_per_group = (
    df.groupby(feature_cols, dropna=False)[TARGET]
      .nunique()
)
print(
    "Kombinasi fitur yang memiliki lebih dari satu label:",
    int((label_counts_per_group > 1).sum())
)

# 3. MENYIAPKAN FITUR DAN TARGET
# ID tidak digunakan sebagai fitur prediksi
X = df[feature_cols].copy()
y = df[TARGET].astype(str).copy()

# Kelompokkan baris dengan kombinasi fitur yang sama.
# Tujuannya agar kombinasi identik tidak masuk ke train dan test sekaligus.
groups = (
    X.astype("string")
     .fillna("<NILAI_KOSONG>")
     .agg(" | ".join, axis=1)
)

# Tentukan kolom numerik dan kategorikal
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
categorical_cols = [
    c for c in X.columns if c not in numeric_cols
]

print("\nKolom numerik:", numeric_cols)
print("Kolom kategorikal:", categorical_cols)

# Preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

# 4. MEMBAGI DATA TANPA KEBOCORAN KOMBINASI IDENTIK
# Cari pembagian yang memiliki semua kelas pada train dan test.
# Jika tidak ditemukan, hentikan evaluasi agar hasil tidak menyesatkan.
splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

classes = set(y.unique())
selected_split = None

for train_idx, test_idx in splitter.split(X, y, groups):
    if (
        set(y.iloc[train_idx].unique()) == classes
        and set(y.iloc[test_idx].unique()) == classes
    ):
        selected_split = (train_idx, test_idx)
        break

if selected_split is None:
    raise ValueError(
        "Tidak ditemukan pembagian 5-fold yang memiliki semua kelas "
        "di train dan test. Periksa jumlah sampel setiap kelas dan "
        "kombinasi fitur. Jangan langsung menggunakan skor evaluasi "
        "biasa karena kombinasi fitur berulang dapat menyebabkan kebocoran."
    )

train_idx, test_idx = selected_split

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("\n" + "=" * 60)
print("3. PEMBAGIAN DATA")
print("=" * 60)
print("Data train:", X_train.shape[0], "baris")
print("Data test :", X_test.shape[0], "baris")
print("Kelas train:", y_train.value_counts().to_dict())
print("Kelas test :", y_test.value_counts().to_dict())

# Verifikasi tidak ada grup fitur yang sama di kedua bagian
assert set(groups.iloc[train_idx]).isdisjoint(
    set(groups.iloc[test_idx])
)
print("Pemeriksaan grup: tidak ada kombinasi fitur identik "
      "di train dan test.")

# 5. MELATIH DAN MEMBANDINGKAN MODEL
models = {
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        class_weight="balanced",
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        class_weight="balanced",
        random_state=42
    )
}

results = []
predictions = {}

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    predictions[name] = y_pred

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Macro F1": f1_score(
            y_test, y_pred, average="macro", zero_division=0
        )
    })

results_df = pd.DataFrame(results).sort_values(
    "Macro F1", ascending=False
)

print("\n" + "=" * 60)
print("4. HASIL PERBANDINGAN MODEL")
print("=" * 60)
display(results_df.round(4))

# 6. LAPORAN DETAIL DAN CONFUSION MATRIX
for name, y_pred in predictions.items():
    print("\n" + "=" * 60)
    print("MODEL:", name)
    print("=" * 60)

    print("\nClassification report:")
    print(classification_report(
        y_test,
        y_pred,
        labels=sorted(classes),
        zero_division=0
    ))

    cm = confusion_matrix(
        y_test, y_pred, labels=sorted(classes)
    )

    cm_df = pd.DataFrame(
        cm,
        index=[f"Aktual: {c}" for c in sorted(classes)],
        columns=[f"Prediksi: {c}" for c in sorted(classes)]
    )

    print("Confusion matrix:")
    display(cm_df)

print("\nSelesai. Perhatikan Macro F1 dan performa tiap kelas.")
print(
    "Hasil ini hanya evaluasi pada dataset yang tersedia, "
    "bukan bukti bahwa model pasti akurat untuk mahasiswa nyata."
)


1. INFORMASI DATASET
Ukuran dataset: (500, 9)

Lima baris pertama:


,ID_Mahasiswa,Jam_Belajar_Harian,Jam_Tidur_Harian,Kehadiran_Kuliah_Pct,IPK_Terakhir,Aktivitas_Organisasi,Tingkat_Stres,Penggunaan_Gadget_Malam_Jam,Produktivitas
0,MHS001,3,7,96,2.89,Tidak,Sedang,1,Tinggi
1,MHS002,8,7,61,3.51,Ya,Rendah,3,Tinggi
2,MHS003,6,6,65,3.12,Ya,Rendah,3,Tinggi
3,MHS004,5,8,73,3.42,Ya,Rendah,3,Tinggi
4,MHS005,2,7,68,2.51,Ya,Sedang,6,Rendah



Jumlah nilai kosong:
ID_Mahasiswa                   0
Jam_Belajar_Harian             0
Jam_Tidur_Harian               0
Kehadiran_Kuliah_Pct           0
IPK_Terakhir                   0
Aktivitas_Organisasi           0
Tingkat_Stres                  0
Penggunaan_Gadget_Malam_Jam    0
Produktivitas                  0
dtype: int64

Distribusi target:
Produktivitas
Sedang    260
Tinggi    163
Rendah     77
Name: count, dtype: int64

Persentase target:
Produktivitas
Sedang    52.0
Tinggi    32.6
Rendah    15.4
Name: proportion, dtype: float64

2. PEMERIKSAAN DUPLIKASI
Jumlah baris: 500
ID unik: 500
Kombinasi fitur unik: 164
Kombinasi fitur yang memiliki lebih dari satu label: 0

Kolom numerik: ['Jam_Belajar_Harian', 'Jam_Tidur_Harian', 'Kehadiran_Kuliah_Pct', 'IPK_Terakhir', 'Penggunaan_Gadget_Malam_Jam']
Kolom kategorikal: ['Aktivitas_Organisasi', 'Tingkat_Stres']

3. PEMBAGIAN DATA
Data train: 367 baris
Data test : 133 baris
Kelas train: {'Sedang': 190, 'Tinggi': 127, 'Rendah': 50}
Kela

,Model,Accuracy,Macro F1
1,Random Forest,1.0000,1.0000
0,Decision Tree,0.9925,0.9929



MODEL: Decision Tree

Classification report:
              precision    recall  f1-score   support

      Rendah       1.00      1.00      1.00        27
      Sedang       0.99      1.00      0.99        70
      Tinggi       1.00      0.97      0.99        36

    accuracy                           0.99       133
   macro avg       1.00      0.99      0.99       133
weighted avg       0.99      0.99      0.99       133

Confusion matrix:


,Prediksi: Rendah,Prediksi: Sedang,Prediksi: Tinggi
Aktual: Rendah,27,0,0
Aktual: Sedang,0,70,0
Aktual: Tinggi,0,1,35



MODEL: Random Forest

Classification report:
              precision    recall  f1-score   support

      Rendah       1.00      1.00      1.00        27
      Sedang       1.00      1.00      1.00        70
      Tinggi       1.00      1.00      1.00        36

    accuracy                           1.00       133
   macro avg       1.00      1.00      1.00       133
weighted avg       1.00      1.00      1.00       133

Confusion matrix:


,Prediksi: Rendah,Prediksi: Sedang,Prediksi: Tinggi
Aktual: Rendah,27,0,0
Aktual: Sedang,0,70,0
Aktual: Tinggi,0,0,36



Selesai. Perhatikan Macro F1 dan performa tiap kelas.
Hasil ini hanya evaluasi pada dataset yang tersedia, bukan bukti bahwa model pasti akurat untuk mahasiswa nyata.


In [2]:

# PEMERIKSAAN LANJUTAN MODEL

from sklearn.inspection import permutation_importance
import pandas as pd

# Ambil model Random Forest yang sudah dilatih dari kode sebelumnya
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        class_weight="balanced",
        random_state=42
    ))
])

rf_pipeline.fit(X_train, y_train)

# 1. Lihat hubungan fitur kategorikal dengan label
for col in ["Tingkat_Stres", "Aktivitas_Organisasi"]:
    if col in df.columns:
        print(f"\nDistribusi Produktivitas berdasarkan {col} (%)")
        tabel = pd.crosstab(
            df[col],
            df[TARGET],
            normalize="index"
        ) * 100
        display(tabel.round(1))

# 2. Hitung permutation importance pada data test
importance = permutation_importance(
    rf_pipeline,
    X_test,
    y_test,
    scoring="f1_macro",
    n_repeats=10,
    random_state=42
)

importance_df = pd.DataFrame({
    "Fitur": X_test.columns,
    "Penurunan_Macro_F1": importance.importances_mean,
    "Simpangan_Baku": importance.importances_std
}).sort_values("Penurunan_Macro_F1", ascending=False)

print("\nPeringkat pengaruh fitur terhadap Macro F1:")
display(importance_df.round(4))



Distribusi Produktivitas berdasarkan Tingkat_Stres (%)


Produktivitas,Rendah,Sedang,Tinggi
Tingkat_Stres,,,
Rendah,0.6,1.9,97.5
Sedang,0.9,96.8,2.3
Tinggi,63.8,36.2,0.0



Distribusi Produktivitas berdasarkan Aktivitas_Organisasi (%)


Produktivitas,Rendah,Sedang,Tinggi
Aktivitas_Organisasi,,,
Tidak,14.6,84.3,1.1
Ya,16.4,12.8,70.8



Peringkat pengaruh fitur terhadap Macro F1:


,Fitur,Penurunan_Macro_F1,Simpangan_Baku
0,Jam_Belajar_Harian,0.0996,0.0116
3,IPK_Terakhir,0.0097,0.0063
1,Jam_Tidur_Harian,0.0000,0.0000
2,Kehadiran_Kuliah_Pct,0.0000,0.0000
4,Aktivitas_Organisasi,0.0000,0.0000
5,Tingkat_Stres,0.0000,0.0000
6,Penggunaan_Gadget_Malam_Jam,0.0000,0.0000


In [3]:
import pandas as pd

# Statistik fitur numerik untuk setiap kategori produktivitas
fitur_numerik = [
    "Jam_Belajar_Harian",
    "Jam_Tidur_Harian",
    "Kehadiran_Kuliah_Pct",
    "IPK_Terakhir",
    "Penggunaan_Gadget_Malam_Jam"
]

print("Rata-rata fitur berdasarkan Produktivitas:")
display(
    df.groupby("Produktivitas")[fitur_numerik]
      .agg(["mean", "median", "min", "max"])
      .round(2)
)

# Periksa apakah jam belajar membedakan kategori secara jelas
print("\nJam belajar berdasarkan Produktivitas:")
display(
    df.groupby("Produktivitas")["Jam_Belajar_Harian"]
      .describe()
      .round(2)
)

Rata-rata fitur berdasarkan Produktivitas:


Jam_Belajar_Harian                Jam_Tidur_Harian             \
                            mean median min max             mean median min   
Produktivitas                                                                 
Rendah                      1.56    2.0   1   2             7.17    7.0   5   
Sedang                      4.03    4.0   3   6             6.22    6.0   5   
Tinggi                      6.67    6.0   3   8             7.51    8.0   6   

                  Kehadiran_Kuliah_Pct                IPK_Terakhir         \
              max                 mean median min max         mean median   
Produktivitas                                                               
Rendah          9                58.56   61.0  52  80         2.21   2.26   
Sedang          9                75.34   76.0  63  90         2.92   2.94   
Tinggi          9                87.82   86.0  61  96         3.54   3.42   

                          Penggunaan_Gadget_Malam_Jam                 
                min   max                        mean median min max  
Produktivitas                                                         
Rendah         2.05  2.60                        5.43    5.0   4   6  
Sedang         2.45  3.32                        3.32    3.0   2   6  
Tinggi         2.89  3.88                        1.63    2.0   1   4


Jam belajar berdasarkan Produktivitas:


,count,mean,std,min,25%,50%,75%,max
Produktivitas,,,,,,,,
Rendah,77.0,1.56,0.50,1.0,1.0,2.0,2.0,2.0
Sedang,260.0,4.03,0.84,3.0,3.0,4.0,5.0,6.0
Tinggi,163.0,6.67,0.88,3.0,6.0,6.0,7.0,8.0


In [4]:
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.base import clone
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

X = df[feature_cols]
y = df[TARGET]

# Kelompokkan baris dengan kombinasi fitur yang sama
groups = X.astype(str).agg("|".join, axis=1)

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

hasil = []

for fold, (train_idx, test_idx) in enumerate(
    cv.split(X, y, groups=groups), start=1
):
    model = clone(rf_pipeline)

    X_train_cv = X.iloc[train_idx]
    X_test_cv = X.iloc[test_idx]
    y_train_cv = y.iloc[train_idx]
    y_test_cv = y.iloc[test_idx]

    model.fit(X_train_cv, y_train_cv)
    pred = model.predict(X_test_cv)

    hasil.append({
        "Fold": fold,
        "Accuracy": accuracy_score(y_test_cv, pred),
        "Macro_F1": f1_score(
            y_test_cv, pred, average="macro"
        )
    })

hasil_df = pd.DataFrame(hasil)

display(hasil_df.round(4))
print("\nRata-rata skor:")
display(hasil_df[["Accuracy", "Macro_F1"]].mean().to_frame("Rata-rata").round(4))

print("\nSimpangan baku skor:")
display(hasil_df[["Accuracy", "Macro_F1"]].std().to_frame("Simpangan Baku").round(4))

,Fold,Accuracy,Macro_F1
0,1,1.0000,1.0000
1,2,0.9817,0.9623
2,3,0.9833,0.9876
3,4,0.9917,0.9930
4,5,0.9500,0.9633



Rata-rata skor:


,Rata-rata
Accuracy,0.9813
Macro_F1,0.9812



Simpangan baku skor:


,Simpangan Baku
Accuracy,0.0190
Macro_F1,0.0174


In [5]:
import joblib
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

# Gunakan fitur tanpa ID dan tanpa kolom target
X = df[feature_cols]
y = df[TARGET]

# Bangun pipeline final
final_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        class_weight="balanced",
        random_state=42
    ))
])

# Latih pada seluruh dataset
final_model.fit(X, y)

# Simpan model dan nama fitur
artifact = {
    "model": final_model,
    "feature_cols": feature_cols,
    "target_name": TARGET,
    "classes": list(final_model.named_steps["model"].classes_)
}

joblib.dump(artifact, "student_productivity_model.joblib")

print("Model berhasil disimpan.")
print("Fitur:", feature_cols)
print("Kelas:", artifact["classes"])

Model berhasil disimpan.
Fitur: ['Jam_Belajar_Harian', 'Jam_Tidur_Harian', 'Kehadiran_Kuliah_Pct', 'IPK_Terakhir', 'Aktivitas_Organisasi', 'Tingkat_Stres', 'Penggunaan_Gadget_Malam_Jam']
Kelas: ['Rendah', 'Sedang', 'Tinggi']
